In [1]:
from bert import Bert
from transformers import BertTokenizer
import torch

/opt/anaconda3/envs/diss/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = Bert('bert-base-uncased')

In [3]:
text = "This is a test sentence for BERT model."
batch = tokenizer(text, return_tensors='pt')
input_ids = batch['input_ids']
attention_mask = batch['attention_mask']

In [15]:
input_ids.shape

torch.Size([1, 11])

In [27]:
hidden1 = model.get_embedding_at_layer(input_ids, attention_mask, 6)
hidden2 = model.get_embedding_at_layer(input_ids, attention_mask, 6)
hidden3 = model.get_embedding_at_layer(input_ids, attention_mask, 6)
hidden4 = model.get_embedding_at_layer(input_ids, attention_mask, 6)

In [63]:
all_hidden = torch.concat([hidden1, hidden2, hidden3, hidden4], dim=0)
all_mask = torch.ones_like(all_hidden[:, :, 0])  # Assuming all hidden states have the same shape

In [64]:
all_hidden.shape, all_mask.shape

(torch.Size([4, 11, 768]), torch.Size([4, 11]))

In [66]:
output = model.forward_from_layer(all_hidden, all_mask, 7)

/opt/anaconda3/envs/diss/lib/python3.10/site-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [ ]:
output

torch.Size([4, 2])

In [1]:
import torch.nn.functional as F
from bert import Bert
from transformers import BertTokenizer
import torch

def test_forward_mixup():
    device = 'mps'
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = Bert('bert-base-uncased').to(device)
    model.train()

    # Create toy input
    sentences = ["hello world"] * 8
    encoded = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True)
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    
    labels = torch.randint(0, 2, (8,)).to(device)
    labels = F.one_hot(labels, num_classes=2).float()

    # Split into 4 batches of 2
    logits, mixed_labels = model.forward_mixup(
        input_ids[:2], attention_mask[:2],
        input_ids[2:4], attention_mask[2:4],
        labels[:2],
        input_ids[4:6], attention_mask[4:6],
        input_ids[6:8], attention_mask[6:8],
        labels[4:6],
        layer_index=7,  # Mix after 7 layers
        mix_lambda=0.7,
        device=device
    )

    print("Logits shape:", logits.shape)
    print("Mixed labels shape:", mixed_labels.shape)

    # Check backward
    loss = torch.nn.BCEWithLogitsLoss()(logits, mixed_labels)
    loss.backward()
    print("Backward pass successful. Loss:", loss.item())

/opt/anaconda3/envs/diss/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
test_forward_mixup()

/opt/anaconda3/envs/diss/lib/python3.10/site-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Logits shape: torch.Size([8, 2])
Mixed labels shape: torch.Size([8, 2])
Backward pass successful. Loss: 0.8629825115203857
